# 08e Stable-Match Anchor Lane Postprocess v1

08d의 anchor-offset target path 아이디어는 유지하되, single lane을 무조건 left/right로 확정하지 않는다.

08e의 핵심은 다음과 같다.

```text
1. both lane이 안정적으로 보일 때 left/right lane geometry를 memory에 저장
2. single lane이 보이면 이전 left/right 중 어느 lane과 이어지는지 matching
3. match cost와 margin이 충분할 때만 single_anchor로 사용
4. 애매하면 uncertain_hold로 보내고 새 target path를 만들지 않음
5. lane이 모두 사라지면 lost_hold/lost_search
```

즉, 조향식은 여전히 `center_error + heading_error`지만, 한쪽 lane의 side 판단은 이전 stable pair와의 시간적 연속성으로 검증한다.


In [ ]:
from __future__ import annotations

import csv
import json
from collections import Counter
from pathlib import Path

import cv2
import numpy as np


BASE = Path(r"~\02_Projects\University\26-1_EmbeddedArtificialSystemOptimization\10_experiments\12_clrkdnet_supervised_rebuild")
REVIEW_ROOT = BASE / "review_outputs" / "08e_stable_match_anchor_postprocess_v1"
VIDEO_DIR = REVIEW_ROOT / "videos"
TABLE_DIR = REVIEW_ROOT / "tables"
CONFIG_DIR = REVIEW_ROOT / "config"
FRAME_REVIEW_DIR = REVIEW_ROOT / "frame_review"
for d in [REVIEW_ROOT, VIDEO_DIR, TABLE_DIR, CONFIG_DIR, FRAME_REVIEW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PKG10 = BASE / "review_outputs" / "10_pi_runtime_latency_sequence_validation_v1" / "pkg"
RECORDS_CSV = PKG10 / "t" / "records_manifest.csv"
DECODED_JSONL = PKG10 / "r" / "ref_decoded.jsonl"

RAW_W = 1296
RAW_H = 972
CUT_HEIGHT = 445
IMAGE_CENTER_X = RAW_W / 2.0
HALF_W = RAW_W / 2.0

FIELD3_VIDEO_PATH = VIDEO_DIR / "field3_08e_stable_match_anchor_postprocess_v1.mp4"
FIELD3_SEQUENCE_CSV = TABLE_DIR / "field3_08e_stable_match_anchor_postprocess_v1.csv"
CONFIG_JSON = CONFIG_DIR / "lane_behavior_08e_stable_match_anchor_v1.json"


LANE_BEHAVIOR_08E = {
    # y anchors used to read lane geometry from decoder points.
    "near_y_ratio": 0.96,
    "mid_y_ratio": 0.84,
    "far_y_ratio": 0.68,

    # both-lane target path: centerline + heading.
    "both_center_gain": 1.00,
    "both_heading_gain": 0.45,

    # single-lane target path: matched anchor lane + offset.
    "single_offset_ratio": 0.70,
    "anchor_center_gain": 1.10,
    "anchor_heading_gain": 0.70,

    # stable pair matching gate. If matching is uncertain, do not create a new target path.
    "match_accept_cost": 0.58,
    "match_margin": 0.08,

    # temporal smoothing.
    "memory_alpha": 0.25,
    "steer_alpha_both": 0.60,
    "steer_alpha_anchor": 0.45,
    "steer_alpha_uncertain": 0.25,

    # lost handling.
    "lost_hold_frames": 4,
    "lost_search_boost": 1.10,
    "lost_decay": 0.92,

    # speed scale emitted by lane postprocess.
    "speed_both": 1.00,
    "speed_anchor": 0.55,
    "speed_uncertain": 0.35,
    "speed_lost_hold": 0.32,
    "speed_lost_search": 0.25,

    "max_steer_norm": 0.75,
}


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def lerp(a, b, alpha):
    return (1.0 - alpha) * float(a) + alpha * float(b)


def derive_internal_thresholds(cfg):
    return {
        "near_y": RAW_H * float(cfg["near_y_ratio"]),
        "mid_y": RAW_H * float(cfg["mid_y_ratio"]),
        "far_y": RAW_H * float(cfg["far_y_ratio"]),
        "min_points": 4,
        "min_y_span_px": RAW_H * 0.055,
        "max_interp_gap_px": RAW_H * 0.16,
        "max_extrapolate_px": RAW_H * 0.16,
        "pair_gap_min_px": RAW_W * 0.20,
        "pair_gap_max_px": RAW_W * 0.95,
        "fallback_half_gap_px": RAW_W * 0.28,
    }


INTERNAL_08E = derive_internal_thresholds(LANE_BEHAVIOR_08E)


def fit_x_as_function_of_y(points):
    pts = np.asarray(points, dtype=np.float32)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) < 2:
        return None
    y = pts[:, 1].astype(np.float64)
    x = pts[:, 0].astype(np.float64)
    if float(np.max(y) - np.min(y)) < 1e-6:
        return None
    a, b = np.polyfit(y, x, deg=1)
    return float(a), float(b)


def x_at_y(points, y, internal):
    pts = np.asarray(points, dtype=np.float32)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) < 2:
        return None, "missing"
    order = np.argsort(pts[:, 1])
    xs = pts[order, 0]
    ys = pts[order, 1]
    y = float(y)
    if ys[0] <= y <= ys[-1]:
        idx = int(np.searchsorted(ys, y))
        if idx <= 0:
            return float(xs[0]), "interp"
        if idx >= len(ys):
            return float(xs[-1]), "interp"
        y0, y1 = float(ys[idx - 1]), float(ys[idx])
        x0, x1 = float(xs[idx - 1]), float(xs[idx])
        if abs(y1 - y0) > internal["max_interp_gap_px"]:
            return None, "gap"
        t = 0.0 if abs(y1 - y0) < 1e-6 else (y - y0) / (y1 - y0)
        return float(x0 + t * (x1 - x0)), "interp"

    gap = min(abs(y - float(ys[0])), abs(y - float(ys[-1])))
    if gap <= internal["max_extrapolate_px"]:
        fit = fit_x_as_function_of_y(pts)
        if fit is not None:
            a, b = fit
            return float(a * y + b), "extrap"
    return None, "missing"


def lane_feature(lane, internal):
    points = np.asarray(lane.get("points", []), dtype=np.float32)
    if points.ndim != 2 or points.shape[1] != 2 or len(points) < int(internal["min_points"]):
        return None
    points = points[np.isfinite(points).all(axis=1)]
    if len(points) < int(internal["min_points"]):
        return None
    y_span = float(np.max(points[:, 1]) - np.min(points[:, 1]))
    if y_span < float(internal["min_y_span_px"]):
        return None
    xs, methods = {}, {}
    for name in ["near", "mid", "far"]:
        x, method = x_at_y(points, internal[f"{name}_y"], internal)
        if x is None:
            return None
        xs[name] = float(x)
        methods[name] = method
    extrap_count = sum(1 for m in methods.values() if m == "extrap")
    quality = float(lane.get("conf", 0.0)) - 0.10 * extrap_count
    heading = (xs["far"] - xs["near"]) / HALF_W
    return {
        "x_near": xs["near"],
        "x_mid": xs["mid"],
        "x_far": xs["far"],
        "heading": float(heading),
        "conf": float(lane.get("conf", 0.0)),
        "quality": float(quality),
        "extrap_count": int(extrap_count),
        "methods": methods,
        "y_span": y_span,
        "raw_lane": lane,
    }


def extract_lane_features(lanes, internal):
    feats = []
    for lane in lanes:
        feat = lane_feature(lane, internal)
        if feat is not None:
            feats.append(feat)
    feats.sort(key=lambda f: f["x_mid"])
    return feats


def empty_lane_memory_feature(x_mid, heading=0.0):
    return {
        "x_near": float(x_mid),
        "x_mid": float(x_mid),
        "x_far": float(x_mid),
        "heading": float(heading),
        "conf": 0.0,
        "quality": 0.0,
        "extrap_count": 0,
    }


def init_memory_08e():
    half_gap = INTERNAL_08E["fallback_half_gap_px"]
    return {
        "has_stable_pair": False,
        "left": empty_lane_memory_feature(IMAGE_CENTER_X - half_gap),
        "right": empty_lane_memory_feature(IMAGE_CENTER_X + half_gap),
        "half_gap_near": half_gap,
        "half_gap_mid": half_gap,
        "half_gap_far": half_gap,
        "target_near": IMAGE_CENTER_X,
        "target_mid": IMAGE_CENTER_X,
        "target_far": IMAGE_CENTER_X,
        "heading": 0.0,
        "last_steer_norm": 0.0,
        "last_raw_steer_norm": 0.0,
        "last_mode": "init",
        "lost_frames": 0,
        "both_stable_frames": 0,
        "anchor_stable_frames": 0,
        "uncertain_frames": 0,
    }


def blend_feature(old, new, alpha):
    out = dict(old)
    for key in ["x_near", "x_mid", "x_far", "heading", "conf", "quality"]:
        out[key] = lerp(old[key], new[key], alpha)
    out["extrap_count"] = int(new.get("extrap_count", 0))
    return out


def lane_match_cost(feature, ref):
    dx_near = abs(feature["x_near"] - ref["x_near"]) / HALF_W
    dx_mid = abs(feature["x_mid"] - ref["x_mid"]) / HALF_W
    dx_far = abs(feature["x_far"] - ref["x_far"]) / HALF_W
    dx = 0.25 * dx_near + 0.45 * dx_mid + 0.30 * dx_far
    dh = abs(feature["heading"] - ref["heading"])
    extrap_penalty = 0.08 * int(feature.get("extrap_count", 0))
    conf_bonus = 0.08 * float(feature.get("quality", 0.0))
    return float(dx + 0.55 * dh + extrap_penalty - conf_bonus)


def pair_geometry(left, right):
    target_near = 0.5 * (left["x_near"] + right["x_near"])
    target_mid = 0.5 * (left["x_mid"] + right["x_mid"])
    target_far = 0.5 * (left["x_far"] + right["x_far"])
    return {
        "left": left,
        "right": right,
        "target_near": float(target_near),
        "target_mid": float(target_mid),
        "target_far": float(target_far),
        "half_gap_near": float(0.5 * abs(right["x_near"] - left["x_near"])),
        "half_gap_mid": float(0.5 * abs(right["x_mid"] - left["x_mid"])),
        "half_gap_far": float(0.5 * abs(right["x_far"] - left["x_far"])),
        "heading": float((target_far - target_near) / HALF_W),
        "conf": float(0.5 * (left["conf"] + right["conf"])),
        "quality": float(0.5 * (left["quality"] + right["quality"])),
    }


def valid_pair(g, internal):
    gap_mid = 2.0 * float(g["half_gap_mid"])
    return internal["pair_gap_min_px"] <= gap_mid <= internal["pair_gap_max_px"]


def pair_score(g, memory):
    center_norm = abs((g["target_mid"] - IMAGE_CENTER_X) / HALF_W)
    score = float(g["quality"]) - 0.25 * center_norm
    if memory["has_stable_pair"]:
        mem_center = abs((g["target_mid"] - memory["target_mid"]) / HALF_W)
        mem_heading = abs(g["heading"] - memory["heading"])
        score -= 0.20 * mem_center + 0.12 * mem_heading
        score -= 0.10 * (lane_match_cost(g["left"], memory["left"]) + lane_match_cost(g["right"], memory["right"]))
    return score


def best_pair(features, memory, internal):
    if len(features) < 2:
        return None
    best, best_score = None, -1e9
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            left, right = features[i], features[j]
            if left["x_mid"] >= right["x_mid"]:
                continue
            g = pair_geometry(left, right)
            if not valid_pair(g, internal):
                continue
            s = pair_score(g, memory)
            if s > best_score:
                best, best_score = g, s
    return best


def update_pair_memory(memory, pair, cfg):
    alpha = float(cfg["memory_alpha"])
    if not memory["has_stable_pair"]:
        alpha = 1.0
    memory["has_stable_pair"] = True
    memory["left"] = blend_feature(memory["left"], pair["left"], alpha)
    memory["right"] = blend_feature(memory["right"], pair["right"], alpha)
    memory["half_gap_near"] = lerp(memory["half_gap_near"], pair["half_gap_near"], alpha)
    memory["half_gap_mid"] = lerp(memory["half_gap_mid"], pair["half_gap_mid"], alpha)
    memory["half_gap_far"] = lerp(memory["half_gap_far"], pair["half_gap_far"], alpha)
    memory["target_near"] = lerp(memory["target_near"], pair["target_near"], alpha)
    memory["target_mid"] = lerp(memory["target_mid"], pair["target_mid"], alpha)
    memory["target_far"] = lerp(memory["target_far"], pair["target_far"], alpha)
    memory["heading"] = lerp(memory["heading"], pair["heading"], alpha)


def match_feature_to_memory(feature, memory):
    left_cost = lane_match_cost(feature, memory["left"])
    right_cost = lane_match_cost(feature, memory["right"])
    if left_cost <= right_cost:
        side, best, other = "left", left_cost, right_cost
    else:
        side, best, other = "right", right_cost, left_cost
    return {
        "side": side,
        "best_cost": float(best),
        "left_cost": float(left_cost),
        "right_cost": float(right_cost),
        "margin": float(other - best),
    }


def choose_anchor_by_stable_match(features, memory, cfg):
    if not features or not memory["has_stable_pair"]:
        return None
    candidates = []
    for feat in features:
        match = match_feature_to_memory(feat, memory)
        candidates.append((match["best_cost"], -match["margin"], feat, match))
    candidates.sort(key=lambda x: (x[0], x[1]))
    _, _, feature, match = candidates[0]
    accepted = match["best_cost"] <= float(cfg["match_accept_cost"]) and match["margin"] >= float(cfg["match_margin"])
    if not accepted:
        return {
            "accepted": False,
            "feature": feature,
            "match": match,
            "reason": f"match_rejected:cost={match['best_cost']:.3f}:margin={match['margin']:.3f}",
        }
    return {
        "accepted": True,
        "feature": feature,
        "match": match,
        "reason": f"match_accepted:{match['side']}:cost={match['best_cost']:.3f}:margin={match['margin']:.3f}",
    }


def anchor_target_path(feature, side, memory, cfg):
    ratio = float(cfg["single_offset_ratio"])
    sign = 1.0 if side == "left" else -1.0
    target_near = feature["x_near"] + sign * memory["half_gap_near"] * ratio
    target_mid = feature["x_mid"] + sign * memory["half_gap_mid"] * ratio
    target_far = feature["x_far"] + sign * memory["half_gap_far"] * ratio
    return {
        "target_near": float(target_near),
        "target_mid": float(target_mid),
        "target_far": float(target_far),
        "heading": float((target_far - target_near) / HALF_W),
        "anchor": feature,
        "anchor_side": side,
        "conf": float(feature["conf"]),
        "quality": float(feature["quality"]),
    }


def clip_steer(v, cfg):
    return float(clamp(v, -float(cfg["max_steer_norm"]), float(cfg["max_steer_norm"])))


def target_to_command(target, mode, memory, cfg):
    center_error = (float(target["target_mid"]) - IMAGE_CENTER_X) / HALF_W
    heading_error = float(target["heading"])
    if mode == "both_stable":
        raw = float(cfg["both_center_gain"]) * center_error + float(cfg["both_heading_gain"]) * heading_error
        alpha, speed = float(cfg["steer_alpha_both"]), float(cfg["speed_both"])
    elif mode == "single_anchor":
        raw = float(cfg["anchor_center_gain"]) * center_error + float(cfg["anchor_heading_gain"]) * heading_error
        alpha, speed = float(cfg["steer_alpha_anchor"]), float(cfg["speed_anchor"])
    else:
        raise ValueError(mode)
    raw = clip_steer(raw, cfg)
    steer = clip_steer(lerp(memory["last_steer_norm"], raw, alpha), cfg)
    return {
        "raw_steer_norm": raw,
        "steer_norm": steer,
        "speed_scale": speed,
        "center_error": float(center_error),
        "heading_error": float(heading_error),
    }


def pack_drive(memory, mode, steer, raw, speed, center, heading, target, lane_count, feat_count, reason, anchor_side="unknown", conf=0.0, quality=0.0, match=None):
    return {
        "mode": mode,
        "steer_norm": float(steer),
        "raw_steer_norm": float(raw),
        "speed_scale": float(speed),
        "center_error": float(center),
        "heading_error": float(heading),
        "target_near": float(target["target_near"]),
        "target_mid": float(target["target_mid"]),
        "target_far": float(target["target_far"]),
        "confidence": float(conf),
        "quality": float(quality),
        "lane_count": int(lane_count),
        "feature_count": int(feat_count),
        "lost_frames": int(memory["lost_frames"]),
        "both_stable_frames": int(memory["both_stable_frames"]),
        "anchor_stable_frames": int(memory["anchor_stable_frames"]),
        "uncertain_frames": int(memory["uncertain_frames"]),
        "anchor_side": anchor_side,
        "match_cost": None if match is None else float(match["best_cost"]),
        "match_margin": None if match is None else float(match["margin"]),
        "reason": reason,
    }


def hold_previous(memory, cfg, mode, speed, lane_count, feat_count, reason, match=None):
    alpha = float(cfg["steer_alpha_uncertain"])
    raw = clip_steer(memory["last_raw_steer_norm"], cfg)
    steer = clip_steer(lerp(memory["last_steer_norm"], raw, alpha), cfg)
    memory["last_raw_steer_norm"] = raw
    memory["last_steer_norm"] = steer
    memory["last_mode"] = mode
    target = {"target_near": memory["target_near"], "target_mid": memory["target_mid"], "target_far": memory["target_far"]}
    return pack_drive(memory, mode, steer, raw, speed, 0.0, 0.0, target, lane_count, feat_count, reason, match=match)


def update_drive_08e(lanes, memory, cfg=LANE_BEHAVIOR_08E):
    internal = derive_internal_thresholds(cfg)
    features = extract_lane_features(lanes, internal)
    pair = best_pair(features, memory, internal)
    if pair is not None:
        cmd = target_to_command(pair, "both_stable", memory, cfg)
        update_pair_memory(memory, pair, cfg)
        memory["lost_frames"] = 0
        memory["both_stable_frames"] += 1
        memory["anchor_stable_frames"] = 0
        memory["uncertain_frames"] = 0
        memory["last_steer_norm"] = cmd["steer_norm"]
        memory["last_raw_steer_norm"] = cmd["raw_steer_norm"]
        memory["last_mode"] = "both_stable"
        return pack_drive(memory, "both_stable", cmd["steer_norm"], cmd["raw_steer_norm"], cmd["speed_scale"], cmd["center_error"], cmd["heading_error"], pair, len(lanes), len(features), "valid_pair", "pair", pair["conf"], pair["quality"], {"best_cost": 0.0, "margin": 999.0})

    if features:
        anchor = choose_anchor_by_stable_match(features, memory, cfg)
        if anchor is not None and anchor["accepted"]:
            feature, match = anchor["feature"], anchor["match"]
            target = anchor_target_path(feature, match["side"], memory, cfg)
            cmd = target_to_command(target, "single_anchor", memory, cfg)
            memory["lost_frames"] = 0
            memory["both_stable_frames"] = 0
            memory["anchor_stable_frames"] += 1
            memory["uncertain_frames"] = 0
            memory["target_near"] = target["target_near"]
            memory["target_mid"] = target["target_mid"]
            memory["target_far"] = target["target_far"]
            memory["heading"] = target["heading"]
            memory["last_steer_norm"] = cmd["steer_norm"]
            memory["last_raw_steer_norm"] = cmd["raw_steer_norm"]
            memory["last_mode"] = "single_anchor"
            return pack_drive(memory, "single_anchor", cmd["steer_norm"], cmd["raw_steer_norm"], cmd["speed_scale"], cmd["center_error"], cmd["heading_error"], target, len(lanes), len(features), anchor["reason"], match["side"], target["conf"], target["quality"], match)

        memory["lost_frames"] = 0
        memory["both_stable_frames"] = 0
        memory["anchor_stable_frames"] = 0
        memory["uncertain_frames"] += 1
        reason = anchor["reason"] if anchor is not None else "no_stable_pair_memory_for_anchor"
        match = anchor["match"] if anchor is not None else None
        return hold_previous(memory, cfg, "uncertain_hold", float(cfg["speed_uncertain"]), len(lanes), len(features), reason, match)

    memory["lost_frames"] += 1
    memory["both_stable_frames"] = 0
    memory["anchor_stable_frames"] = 0
    memory["uncertain_frames"] = 0
    if memory["lost_frames"] <= int(cfg["lost_hold_frames"]):
        return hold_previous(memory, cfg, "lost_hold", float(cfg["speed_lost_hold"]), len(lanes), len(features), "no_feature_hold_last_steer")
    memory["last_raw_steer_norm"] = clip_steer(memory["last_raw_steer_norm"] * float(cfg["lost_search_boost"]) * float(cfg["lost_decay"]), cfg)
    return hold_previous(memory, cfg, "lost_search", float(cfg["speed_lost_search"]), len(lanes), len(features), "no_feature_search_previous_direction")


def make_line_lane(x_near, x_far, conf=0.9):
    ys = np.linspace(RAW_H - 1, RAW_H * 0.60, 12, dtype=np.float32)
    xs = np.linspace(float(x_near), float(x_far), len(ys), dtype=np.float32)
    return {"points": np.stack([xs, ys], axis=1), "conf": float(conf)}


memory = init_memory_08e()
for name, lanes in [
    ("straight_pair", [make_line_lane(360, 360), make_line_lane(940, 940)]),
    ("right_pair", [make_line_lane(360, 430), make_line_lane(940, 1010)]),
    ("left_survives", [make_line_lane(370, 440)]),
    ("ambiguous", [make_line_lane(680, 200)]),
    ("lost", []),
]:
    d = update_drive_08e(lanes, memory, LANE_BEHAVIOR_08E)
    print(name, {k: d[k] for k in ["mode", "anchor_side", "steer_norm", "speed_scale", "center_error", "heading_error", "match_cost", "match_margin", "reason"]})


In [ ]:
def imread_bgr_unicode(path):
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img


def load_field3_sequence_records(limit=None):
    rows = []
    with RECORDS_CSV.open("r", encoding="utf-8-sig", newline="") as f:
        for row in csv.DictReader(f):
            if row["set"] == "field3" and row["role"] == "sequence":
                row["order"] = int(row["order"])
                rows.append(row)
    rows.sort(key=lambda r: r["order"])
    return rows if limit is None else rows[: int(limit)]


def load_decoded_lanes_by_key():
    out = {}
    with DECODED_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                out[obj["key"]] = obj.get("lanes", [])
    return out


def lanes_from_jsonable(lanes_json):
    lanes = []
    for lane in lanes_json:
        pts = np.asarray(lane.get("points", []), dtype=np.float32)
        if pts.ndim == 2 and pts.shape[1] == 2 and len(pts) >= 2:
            lanes.append({"points": pts, "conf": float(lane.get("conf", 0.0))})
    return lanes


def draw_lane_polyline(img, points, color=(0, 220, 80), thickness=3):
    pts = np.asarray(points, dtype=np.float32)
    pts = pts[np.isfinite(pts).all(axis=1)]
    if len(pts) < 2:
        return
    pts_i = np.round(pts).astype(np.int32).reshape(-1, 1, 2)
    cv2.polylines(img, [pts_i], False, color, thickness, cv2.LINE_AA)


def draw_text_lines(img, lines, x=24, y=34, line_h=28):
    pad = 10
    width = max(720, max((len(s) for s in lines), default=0) * 12)
    height = line_h * len(lines) + pad * 2
    panel = img.copy()
    cv2.rectangle(panel, (x - pad, y - 24), (x - pad + width, y - 24 + height), (0, 0, 0), -1)
    cv2.addWeighted(panel, 0.55, img, 0.45, 0, img)
    for i, text in enumerate(lines):
        cv2.putText(img, text, (x, y + i * line_h), cv2.FONT_HERSHEY_SIMPLEX, 0.68, (245, 245, 245), 2, cv2.LINE_AA)


def draw_08e_replay_frame(bgr, lanes, drive, frame_index, cfg=LANE_BEHAVIOR_08E, scale_width=960):
    out = bgr.copy()
    internal = derive_internal_thresholds(cfg)
    for lane in lanes:
        draw_lane_polyline(out, lane["points"], color=(0, 220, 80), thickness=4)
    for name, color in [("far", (255, 220, 0)), ("mid", (255, 180, 0)), ("near", (255, 140, 0))]:
        y = int(round(internal[f"{name}_y"]))
        cv2.line(out, (0, y), (RAW_W - 1, y), color, 1, cv2.LINE_AA)
        cv2.putText(out, name, (12, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
    cv2.line(out, (int(IMAGE_CENTER_X), int(CUT_HEIGHT)), (int(IMAGE_CENTER_X), RAW_H - 1), (210, 210, 210), 2, cv2.LINE_AA)
    p_near = (int(round(drive["target_near"])), int(round(internal["near_y"])))
    p_mid = (int(round(drive["target_mid"])), int(round(internal["mid_y"])))
    p_far = (int(round(drive["target_far"])), int(round(internal["far_y"])))
    cv2.line(out, p_near, p_mid, (255, 255, 0), 4, cv2.LINE_AA)
    cv2.line(out, p_mid, p_far, (255, 255, 0), 4, cv2.LINE_AA)
    cv2.circle(out, p_mid, 9, (0, 170, 255), -1, cv2.LINE_AA)
    cv2.line(out, (int(IMAGE_CENTER_X), p_mid[1]), p_mid, (0, 170, 255), 3, cv2.LINE_AA)
    base = (int(IMAGE_CENTER_X), RAW_H - 35)
    raw_tip = (int(round(IMAGE_CENTER_X + float(drive["raw_steer_norm"]) * 420.0)), int(round(internal["mid_y"])))
    steer_tip = (int(round(IMAGE_CENTER_X + float(drive["steer_norm"]) * 420.0)), int(round(internal["far_y"])))
    cv2.arrowedLine(out, base, raw_tip, (180, 120, 180), 2, cv2.LINE_AA, tipLength=0.18)
    cv2.arrowedLine(out, base, steer_tip, (255, 0, 220), 5, cv2.LINE_AA, tipLength=0.20)
    mode_color = {"both_stable": (80, 220, 80), "single_anchor": (0, 220, 255), "uncertain_hold": (0, 165, 255), "lost_hold": (0, 110, 255), "lost_search": (0, 60, 255)}.get(str(drive["mode"]), (255, 255, 255))
    cv2.circle(out, (RAW_W - 34, 34), 16, mode_color, -1, cv2.LINE_AA)
    mc = "none" if drive["match_cost"] is None else f"{drive['match_cost']:.2f}"
    mm = "none" if drive["match_margin"] is None else f"{drive['match_margin']:.2f}"
    lines = [
        f"{frame_index:04d} mode={drive['mode']} side={drive['anchor_side']} lanes={drive['lane_count']} feat={drive['feature_count']}",
        f"steer={drive['steer_norm']:+.3f} raw={drive['raw_steer_norm']:+.3f} speed={drive['speed_scale']:.2f} conf={drive['confidence']:.2f}",
        f"center={drive['center_error']:+.3f} heading={drive['heading_error']:+.3f} bothN={drive['both_stable_frames']} anchN={drive['anchor_stable_frames']} unN={drive['uncertain_frames']}",
        f"match_cost={mc} margin={mm} reason={drive['reason']}",
    ]
    draw_text_lines(out, lines)
    if scale_width and out.shape[1] != scale_width:
        scale = float(scale_width) / float(out.shape[1])
        out = cv2.resize(out, (scale_width, int(round(out.shape[0] * scale))), interpolation=cv2.INTER_AREA)
    return out


def generate_field3_08e_video(limit=None, fps=12.0):
    CONFIG_JSON.write_text(json.dumps(LANE_BEHAVIOR_08E, indent=2, ensure_ascii=False), encoding="utf-8")
    records = load_field3_sequence_records(limit=limit)
    decoded_by_key = load_decoded_lanes_by_key()
    memory = init_memory_08e()
    writer = None
    rows = []
    for idx, rec in enumerate(records):
        bgr = imread_bgr_unicode(PKG10 / rec["image_rel"])
        lanes = lanes_from_jsonable(decoded_by_key.get(rec["key"], []))
        drive = update_drive_08e(lanes, memory, LANE_BEHAVIOR_08E)
        frame = draw_08e_replay_frame(bgr, lanes, drive, idx, LANE_BEHAVIOR_08E, scale_width=960)
        if writer is None:
            h, w = frame.shape[:2]
            writer = cv2.VideoWriter(str(FIELD3_VIDEO_PATH), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
            assert writer.isOpened(), FIELD3_VIDEO_PATH
        writer.write(frame)
        rows.append({
            "frame_index": idx,
            "key": rec["key"],
            "source_name": rec["source_name"],
            "mode": drive["mode"],
            "anchor_side": drive["anchor_side"],
            "lane_count": drive["lane_count"],
            "feature_count": drive["feature_count"],
            "steer_norm": float(drive["steer_norm"]),
            "raw_steer_norm": float(drive["raw_steer_norm"]),
            "speed_scale": float(drive["speed_scale"]),
            "center_error": float(drive["center_error"]),
            "heading_error": float(drive["heading_error"]),
            "target_near": float(drive["target_near"]),
            "target_mid": float(drive["target_mid"]),
            "target_far": float(drive["target_far"]),
            "confidence": float(drive["confidence"]),
            "quality": float(drive["quality"]),
            "lost_frames": int(drive["lost_frames"]),
            "both_stable_frames": int(drive["both_stable_frames"]),
            "anchor_stable_frames": int(drive["anchor_stable_frames"]),
            "uncertain_frames": int(drive["uncertain_frames"]),
            "match_cost": "" if drive["match_cost"] is None else float(drive["match_cost"]),
            "match_margin": "" if drive["match_margin"] is None else float(drive["match_margin"]),
            "reason": drive["reason"],
        })
    if writer is not None:
        writer.release()
    with FIELD3_SEQUENCE_CSV.open("w", encoding="utf-8-sig", newline="") as f:
        writer_csv = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["frame_index"])
        writer_csv.writeheader()
        writer_csv.writerows(rows)
    mode_counts = Counter(row["mode"] for row in rows)
    side_counts = Counter(row["anchor_side"] for row in rows)
    steer_abs = np.asarray([abs(row["steer_norm"]) for row in rows], dtype=np.float32)
    raw_abs = np.asarray([abs(row["raw_steer_norm"]) for row in rows], dtype=np.float32)
    speed = np.asarray([row["speed_scale"] for row in rows], dtype=np.float32)
    summary = {
        "frames": len(rows),
        "video": str(FIELD3_VIDEO_PATH),
        "table": str(FIELD3_SEQUENCE_CSV),
        "config": str(CONFIG_JSON),
        "mode_counts": dict(mode_counts),
        "side_counts": dict(side_counts),
        "mean_abs_steer": float(steer_abs.mean()) if len(steer_abs) else None,
        "p90_abs_steer": float(np.percentile(steer_abs, 90)) if len(steer_abs) else None,
        "max_abs_steer": float(steer_abs.max()) if len(steer_abs) else None,
        "p90_abs_raw_steer": float(np.percentile(raw_abs, 90)) if len(raw_abs) else None,
        "mean_speed_scale": float(speed.mean()) if len(speed) else None,
        "min_speed_scale": float(speed.min()) if len(speed) else None,
    }
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    return summary


field3_08e_video_summary = generate_field3_08e_video(limit=None, fps=12.0)


In [ ]:
def write_image_unicode(path, img):
    ok, buf = cv2.imencode(".jpg", img, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    if not ok:
        raise RuntimeError(path)
    buf.tofile(str(path))


def make_frame_review_sheets():
    rows = []
    with FIELD3_SEQUENCE_CSV.open("r", encoding="utf-8-sig", newline="") as f:
        for r in csv.DictReader(f):
            r["frame_index"] = int(r["frame_index"])
            for k in ["steer_norm", "raw_steer_norm", "speed_scale", "center_error", "heading_error"]:
                r[k] = float(r[k])
            rows.append(r)
    interval = list(range(0, len(rows), 20))
    if rows[-1]["frame_index"] not in interval:
        interval.append(rows[-1]["frame_index"])
    transitions, prev = [], None
    for r in rows:
        key = (r["mode"], r["anchor_side"])
        if prev is not None and key != prev:
            transitions.append(r["frame_index"])
        prev = key
    if len(transitions) > 18:
        transitions = [transitions[int(i * (len(transitions) - 1) / 17)] for i in range(18)]
    by_abs = sorted(rows, key=lambda r: abs(r["steer_norm"]), reverse=True)[:12]
    by_center = sorted(rows, key=lambda r: abs(r["center_error"]), reverse=True)[:12]
    problem = sorted(set([r["frame_index"] for r in by_abs + by_center] + transitions))
    cap = cv2.VideoCapture(str(FIELD3_VIDEO_PATH))

    def read_frame(idx):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, frame = cap.read()
        if not ok:
            raise RuntimeError(idx)
        return frame

    def make_sheet(indices, out_path, title, cols=3):
        imgs = []
        for idx in indices:
            frame = cv2.resize(read_frame(idx), (480, 360), interpolation=cv2.INTER_AREA)
            r = rows[idx]
            top = f"#{idx} {r['mode']} {r['anchor_side']} steer={r['steer_norm']:+.2f} spd={r['speed_scale']:.2f}"
            bot = f"center={r['center_error']:+.2f} heading={r['heading_error']:+.2f} reason={r['reason'][:34]}"
            cv2.rectangle(frame, (0, 0), (480, 34), (0, 0, 0), -1)
            cv2.putText(frame, top, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.rectangle(frame, (0, 326), (480, 360), (0, 0, 0), -1)
            cv2.putText(frame, bot, (8, 350), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 2, cv2.LINE_AA)
            imgs.append(frame)
        rows_img = []
        for i in range(0, len(imgs), cols):
            chunk = imgs[i:i + cols]
            while len(chunk) < cols:
                chunk.append(np.zeros_like(imgs[0]))
            rows_img.append(np.hstack(chunk))
        sheet = np.vstack(rows_img)
        header = np.full((50, sheet.shape[1], 3), 245, dtype=np.uint8)
        cv2.putText(header, title, (12, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 20, 20), 2, cv2.LINE_AA)
        write_image_unicode(out_path, np.vstack([header, sheet]))

    make_sheet(interval[:15], FRAME_REVIEW_DIR / "interval_sheet.jpg", "08e interval frames")
    make_sheet(problem[:18], FRAME_REVIEW_DIR / "problem_transition_extreme_sheet.jpg", "08e transition / extreme frames")
    cap.release()
    print("wrote", FRAME_REVIEW_DIR / "interval_sheet.jpg")
    print("wrote", FRAME_REVIEW_DIR / "problem_transition_extreme_sheet.jpg")


make_frame_review_sheets()


## 판단 포인트

08e 영상에서 먼저 볼 것은 다음이다.

- `single_anchor`가 08d처럼 무조건 나오지 않는가?
- 애매한 single lane에서 `uncertain_hold`가 잘 막아주는가?
- `both_stable`로 복귀할 때 target path가 즉시 중앙선으로 돌아오는가?
- `both_stable_frames`, `anchor_stable_frames`, `uncertain_frames`가 상태머신 복귀 조건으로 쓸 만큼 자연스러운가?
